## 02 — Filtering Features

So far every feature in the file appears on the map. Filtering changes that — you decide **which features to show** before the map ever renders.

```
full dataset  →  filter  →  subset  →  map
```

This is where the map stops being a static picture of a file and starts being a **view into your data**.

## Setup

In [ ]:
import json
from ipyleaflet import Map, GeoJSON

WICHITA_FALLS = (33.9137, -98.4934)

with open("../02-Viewing_GeoJSON/data/wichita_falls.geojson") as f:
    geojson = json.load(f)

print(f"{len(geojson['features'])} features loaded")
for feat in geojson["features"]:
    print(f"  {feat['geometry']['type']:12}  {feat['properties']['name']}")

## Simple Property Filter

Filtering is a list comprehension with an `if` condition. The result is a plain Python list of feature dicts.

```python
filtered = [
    f for f in geojson["features"]
    if <condition on f["properties"]>
]
```

In [ ]:
# Keep only features whose type is "park"
park_features = [
    f for f in geojson["features"]
    if f["properties"].get("type") == "park"
]

print(f"{len(park_features)} park features:")
for f in park_features:
    print(f"  {f['properties']['name']}")

## Rebuilding a FeatureCollection

A filtered list is just a list — not a valid GeoJSON object. To pass it to `GeoJSON()`, wrap it back into a FeatureCollection dict.

```python
filtered_geojson = {
    "type": "FeatureCollection",
    "features": filtered_list
}
```

This is the pattern you will use every time you filter.

In [ ]:
park_fc = {
    "type": "FeatureCollection",
    "features": park_features
}

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=park_fc))
m

## Filtering by Geometry Type

You can also filter on the `geometry` side of the feature — not just `properties`. This is useful when you want to control how different geometry types are displayed.

In [ ]:
# Keep only Point features
point_fc = {
    "type": "FeatureCollection",
    "features": [
        f for f in geojson["features"]
        if f["geometry"]["type"] == "Point"
    ]
}

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=point_fc))
m

## Numeric Filtering

When a property holds a number, use comparison operators (`>`, `<`, `>=`, `<=`) in the filter condition.

Our `wichita_falls.geojson` does not have numeric properties, so here is a small inline dataset with a `priority` score. The pattern is identical for any real dataset.

In [ ]:
# Inline dataset with a numeric "priority" property
sites = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.49, 33.91]}, "properties": {"name": "Alpha",   "priority": 92}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.51, 33.89]}, "properties": {"name": "Bravo",   "priority": 45}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.53, 33.87]}, "properties": {"name": "Charlie", "priority": 78}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.50, 33.93]}, "properties": {"name": "Delta",   "priority": 15}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.47, 33.88]}, "properties": {"name": "Echo",    "priority": 61}},
    ]
}

# Only show high-priority sites (score >= 60)
high_priority = {
    "type": "FeatureCollection",
    "features": [
        f for f in sites["features"]
        if f["properties"]["priority"] >= 60
    ]
}

print("High-priority sites:")
for f in high_priority["features"]:
    print(f"  {f['properties']['name']}  ({f['properties']['priority']})")

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=high_priority))
m

## Combining Filtering and Styling

Filtering and styling are independent steps — filter first, then style. The pipeline is:

```
raw GeoJSON
    ↓  filter (list comprehension)
subset list
    ↓  wrap into FeatureCollection
filtered GeoJSON
    ↓  GeoJSON(data=..., style=...)
styled map layer
```

In [ ]:
COLOR_MAP = {
    "park":       "#2ecc71",
    "water":      "#1abc9c",
    "education":  "#3498db",
    "government": "#e74c3c",
    "route":      "#e67e22",
}

def style_by_type(feature):
    color = COLOR_MAP.get(feature["properties"].get("type"), "#95a5a6")
    return {"color": color, "fillColor": color, "fillOpacity": 0.5, "weight": 2}

# Filter: only non-route features
no_routes_fc = {
    "type": "FeatureCollection",
    "features": [
        f for f in geojson["features"]
        if f["properties"].get("type") != "route"
    ]
}

# Style and display
m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=no_routes_fc, style=style_by_type))
m

## Filtering into Multiple Layers

You can filter the same dataset multiple ways to create distinct named layers. Combined with `LayersControl`, this gives the user a toggle for each group.

In [ ]:
from ipyleaflet import LayersControl

def make_fc(features):
    return {"type": "FeatureCollection", "features": features}

natural_layer = GeoJSON(
    data=make_fc([f for f in geojson["features"] if f["properties"].get("type") in ("park", "water")]),
    name="Natural Features",
    style={"color": "#27ae60", "fillColor": "#27ae60", "fillOpacity": 0.45, "weight": 2}
)

built_layer = GeoJSON(
    data=make_fc([f for f in geojson["features"] if f["properties"].get("type") in ("government", "education")]),
    name="Built Environment",
    style={"color": "#2980b9", "fillColor": "#2980b9", "fillOpacity": 0.45, "weight": 2}
)

route_layer = GeoJSON(
    data=make_fc([f for f in geojson["features"] if f["properties"].get("type") == "route"]),
    name="Routes",
    style={"color": "#e67e22", "weight": 3, "opacity": 0.8}
)

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(natural_layer)
m.add(built_layer)
m.add(route_layer)
m.add(LayersControl())
m

Each filter produces one layer. The `LayersControl` lets you toggle each group independently.

## Advanced Filtering: Compound Conditions

Use `and` / `or` to combine multiple conditions in one filter.

```python
# AND — both conditions must be true
f["properties"]["type"] == "park" and f["geometry"]["type"] == "Polygon"

# OR — either condition is enough
f["properties"]["type"] == "park" or f["properties"]["type"] == "water"

# IN — cleaner way to write OR against a fixed set
f["properties"]["type"] in ("park", "water")
```

In [ ]:
# Points that are either parks or water
natural_points = {
    "type": "FeatureCollection",
    "features": [
        f for f in geojson["features"]
        if f["geometry"]["type"] == "Point"
        and f["properties"].get("type") in ("park", "water")
    ]
}

print(f"{len(natural_points['features'])} natural point features:")
for f in natural_points["features"]:
    print(f"  {f['properties']['name']}  ({f['properties']['type']})")

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=natural_points))
m

In [ ]:
# Compound numeric + string filter on the sites dataset
high_and_early = {
    "type": "FeatureCollection",
    "features": [
        f for f in sites["features"]
        if f["properties"]["priority"] >= 60
        and f["properties"]["name"][0] <= "C"  # names starting A–C
    ]
}

print("High-priority, early-alphabet sites:")
for f in high_and_early["features"]:
    print(f"  {f['properties']['name']}  ({f['properties']['priority']})")

Compound conditions are the foundation of **spatial queries** — once you add geometry operations (distance, containment, intersection), you will combine them here the same way.

## Exercise A

Filter `wichita_falls.geojson` to show only `Polygon` and `LineString` features (exclude all `Point` features). Apply `style_by_type` from the teaching cells and display the result.

In [ ]:
from ipyleaflet import Map, GeoJSON

COLOR_MAP = {
    "park":       "#2ecc71",
    "water":      "#1abc9c",
    "education":  "#3498db",
    "government": "#e74c3c",
    "route":      "#e67e22",
}

def style_by_type(feature):
    color = COLOR_MAP.get(feature["properties"].get("type"), "#95a5a6")
    return {"color": color, "fillColor": color, "fillOpacity": 0.5, "weight": 2}

# Filter to only Polygon and LineString features (exclude Points)
# Apply style_by_type and display
# Your code here

## Exercise B

Load `meteorites.geojson` (hint: it's two levels up in `data/`). Filter to meteorites where **mass > 10000 and year is recorded** (not `None` or `0`). Display them on a world map and print the count.

In [ ]:
import json
from ipyleaflet import Map, GeoJSON

with open("../../data/meteorites.geojson") as f:
    meteorites = json.load(f)

# Filter: mass > 10000 AND year is not None/0
# Display on a world map (center=(20, 0), zoom=2), print the count
# Your code here

---

## Check Your Understanding

Using `geojson["features"]`, build a map with **two named layers**:

- `"Points"` — all Point features, styled blue (`#2980b9`)
- `"Areas"` — all Polygon features, styled green (`#27ae60`), `fillOpacity` 0.4

Add a `LayersControl` so both can be toggled.

```python
# your answer here
```

## Next

In [04 — Interactive Maps](../04-Interactive_Maps/00-Map_Events.ipynb), we add event handling — letting users click the map and drive what the map shows.